In [ ]:
import random
import time

In [ ]:
class MaxCliqueTabuSearch:
    def __init__(self):
        self.neighbour_sets = []
        self.non_neighbours = []
        self.best_clique = set()
        self.qco = []
        self.index = []
        self.q_border = 0
        self.c_border = 0

    @staticmethod
    def get_random(a, b):
        return random.randint(a, b)

    def read_graph_file(self, filename):
        with open(filename, 'r') as fin:
            vertices = 0
            edges = 0
            for line in fin:
                line = line.strip()
                if not line:
                    continue
                if line[0] == 'c':
                    continue
                parts = line.split()
                if line[0] == 'p':
                    _, _type, vertices, edges = parts
                    vertices = int(vertices)
                    edges = int(edges)
                    self.neighbour_sets = [set() for _ in range(vertices)]
                    self.non_neighbours = [set() for _ in range(vertices)]
                    self.qco = [0] * vertices
                    self.index = list(range(vertices))
                else:
                    _, start, finish = parts
                    start = int(start) - 1
                    finish = int(finish) - 1
                    self.neighbour_sets[start].add(finish)
                    self.neighbour_sets[finish].add(start)
        num_vertices = len(self.neighbour_sets)
        for i in range(num_vertices):
            for j in range(num_vertices):
                if i != j and j not in self.neighbour_sets[i]:
                    self.non_neighbours[i].add(j)

    def run_search(self, starts, randomization):
        num_vertices = len(self.neighbour_sets)
        for _ in range(starts):
            self.clear_clique()
            for i in range(num_vertices):
                self.qco[i] = i
                self.index[i] = i
            self.run_initial_heurstic(randomization)
            self.c_border = self.q_border
            swaps = 0
            while swaps < 100:
                if not self.move():
                    if not self.swap1to1():
                        break
                    else:
                        swaps += 1
            if self.q_border > len(self.best_clique):
                self.best_clique = set(self.qco[:self.q_border])

    def get_clique(self):
        return self.best_clique

    def check(self):
        for i in self.best_clique:
            for j in self.best_clique:
                if i != j and j not in self.neighbour_sets[i]:
                    print("Returned subgraph is not clique")
                    return False
        return True

    def clear_clique(self):
        self.q_border = 0
        self.c_border = 0

    def compute_tightness(self, vertex):
        tightness = 0
        for i in range(self.q_border):
            if vertex in self.non_neighbours[self.qco[i]]:
                tightness += 1
        return tightness

    def swap_vertices(self, vertex, border):
        idx1 = self.index[vertex]
        idx2 = border
        vertex_at_border = self.qco[idx2]
        self.qco[idx1], self.qco[idx2] = self.qco[idx2], self.qco[idx1]
        self.index[vertex] = idx2
        self.index[vertex_at_border] = idx1

    def insert_to_clique(self, i):
        for j in self.non_neighbours[i]:
            if self.compute_tightness(j) == 0:
                self.c_border -= 1
                self.swap_vertices(j, self.c_border)
        self.swap_vertices(i, self.q_border)
        self.q_border += 1

    def remove_from_clique(self, k):
        for j in self.non_neighbours[k]:
            if self.compute_tightness(j) == 1:
                self.swap_vertices(j, self.c_border)
                self.c_border += 1
        self.q_border -= 1
        self.swap_vertices(k, self.q_border)

    def swap1to1(self):
        for counter in range(self.q_border):
            vertex = self.qco[counter]
            for i in self.non_neighbours[vertex]:
                if self.compute_tightness(i) == 1:
                    self.remove_from_clique(vertex)
                    self.insert_to_clique(i)
                    return True
        return False

    def move(self):
        if self.c_border == self.q_border:
            return False
        vertex = self.qco[self.q_border]
        self.insert_to_clique(vertex)
        return True

    def run_initial_heurstic(self, randomization):
        candidates = list(range(len(self.neighbour_sets)))
        random.shuffle(candidates)
        while candidates:
            last = len(candidates) - 1
            r = self.get_random(0, min(randomization - 1, last))
            vertex = candidates[r]
            self.swap_vertices(vertex, self.q_border)
            self.q_border += 1
            candidates = [c for c in candidates if c in self.neighbour_sets[vertex]]
            random.shuffle(candidates)

In [1]:
iterations = int(input("Number of iterations: "))
randomization = int(input("Randomization: "))

files = [
    "brock200_1.clq", "brock200_2.clq", "brock200_3.clq", "brock200_4.clq", "brock400_1.clq",
    "brock400_2.clq", "brock400_3.clq", "brock400_4.clq", "C125.9.clq", "gen200_p0.9_44.clq",
    "gen200_p0.9_55.clq", "hamming8-4.clq", "johnson16-2-4.clq", "johnson8-2-4.clq", "keller4.clq",
    "MANN_a27.clq", "MANN_a9.clq", "p_hat1000-1.clq", "p_hat1000-2.clq", "p_hat1500-1.clq",
    "p_hat300-3.clq", "p_hat500-3.clq", "san1000.clq", "sanr200_0.9.clq", "sanr400_0.7.clq"
]

with open("clique_tabu.csv", "w") as fout:
    fout.write("File; Clique; Time (sec); Clique vertices\n")
    for file in files:
        problem = MaxCliqueTabuSearch()
        problem.read_graph_file(file)
        start_time = time.time()
        problem.run_search(iterations, randomization)
        elapsed = time.time() - start_time
        if not problem.check():
            print("*** WARNING: incorrect clique ***")
            fout.write("*** WARNING: incorrect clique ***\n")
        clique = problem.get_clique()
        clique_size = len(clique)
        clique_sorted = sorted(clique)
        clique_str = ", ".join(str(v + 1) for v in clique_sorted)
        fout.write(f"{file}; {clique_size}; {elapsed:.4f}; {clique_str}\n")
        print(f"{file}, result - {clique_size}, time - {elapsed:.4f}")
        print(f"Clique vertices: {clique_str}")

Number of iterations: 1000
Randomization: 1
brock200_1.clq, result - 20, time - 44.2758
Clique vertices: 15, 19, 20, 27, 35, 60, 69, 79, 126, 130, 143, 149, 152, 155, 159, 175, 189, 190, 192, 199
brock200_2.clq, result - 12, time - 44.2555
Clique vertices: 27, 48, 55, 70, 105, 120, 121, 135, 145, 149, 158, 183
brock200_3.clq, result - 14, time - 46.7908
Clique vertices: 26, 28, 41, 53, 83, 129, 151, 154, 155, 172, 182, 190, 191, 193
brock200_4.clq, result - 17, time - 48.2734
Clique vertices: 12, 19, 28, 29, 38, 54, 65, 71, 79, 93, 117, 127, 139, 161, 165, 186, 192
brock400_1.clq, result - 24, time - 100.2852
Clique vertices: 11, 13, 31, 40, 76, 78, 125, 127, 135, 137, 161, 186, 199, 203, 222, 226, 232, 306, 310, 320, 351, 360, 381, 394
brock400_2.clq, result - 24, time - 101.8807
Clique vertices: 11, 16, 19, 57, 77, 78, 95, 98, 99, 118, 129, 130, 131, 136, 182, 209, 227, 286, 299, 316, 350, 356, 382, 399
brock400_3.clq, result - 24, time - 100.7571
Clique vertices: 26, 47, 97, 109, 16